In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import json

In [ ]:
with open("../config/settings.json", 'r') as f:
    settings = json.load(f)
    
START_DATE = settings["START_DATE"]
END_DATE = settings["END_DATE"]
INTERVAL_DUR = settings["INTERVAL_DUR"]

In [ ]:
# --- Fetch historical prices covering the full date range ---
# pad start slightly to make sure yfinance has data before the first target date
prices = yf.download(
    "VWRD.L",
    start=pd.Timestamp(START_DATE)-pd.Timedelta(days=10),
    end=pd.Timestamp(END_DATE),
    interval='1d',
    progress=False,
    auto_adjust=True,
)['Close']

#prices = prices.rename('PRICE').reset_index()
#prices = prices.rename(columns={'Date': 'DATE'})

In [ ]:
prices.index.rename("DATE", inplace=True)
prices.index = prices.index.astype('datetime64[us]')
#np.dtype(prices.index)
prices

In [ ]:
# Create date range from start to end date 
date_range = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq=INTERVAL_DUR, 
)
date_series = pd.DataFrame({'DATE': date_range})

# Map cumulative data onto date_range, backwards filling any interval without activity
result = pd.merge_asof(date_series, prices, on='DATE', direction='backward')
result.set_index("DATE", inplace=True)
result

In [ ]:
info = yf.Ticker("VWRD.L").info

In [ ]:
info.get('currency')

In [ ]:
prices = yf.download(
    "VWRD.L",
    start=pd.Timestamp(START_DATE)-pd.Timedelta(days=10),
    end=pd.Timestamp(END_DATE),
    interval='1d',
    progress=False,
    auto_adjust=True,
)['Close']

fx = yf.download(
    "USDEUR=X",
    start=pd.Timestamp(START_DATE) - pd.Timedelta(days=10),
    end=pd.Timestamp(END_DATE) + pd.Timedelta(days=1),
    interval='1d',
    progress=False,
    auto_adjust=True,
)['Close']

In [ ]:
prices["PRICE_EUR"] = prices["VWRD.L"] * fx["USDEUR=X"]
prices